In [ ]:
import pandas as pd
import numpy as np

Further Data Preparation. by Nathan

In [ ]:
df=pd.read_csv('OSMI_survey.csv', delimiter=';')
df.head(5)


Loading the dataset, Use your ablsolute path in place of (OSMI_survey.csv) this ensures that file is accessed if it is not saved in same folder as the jupyter notebook.

use function .head() to see the top 5 row

In [ ]:
# initial data exploration
df.info()

In [ ]:
#removing comments column
#column has too many null values and is not needed for analysis
df_clean=df.drop(columns=['comments'])
df_clean.info()

Here the comments column is dropped. this is becasue only 164/1259 are non null so all results collected from its analysis would be  negligble as they are not generalisable.

In [ ]:
no_employeebins=df_clean['no_employees'].unique()
print(no_employeebins) #un comment to see unique values in the no_employees column
#replacing 'WTD' with the mean of the values between 1 and 25 as we have 1-WTN and WTD-25 in the data
#WTD stands for "working time directive" and is used in the UK to indicate a standard working week
wtd_n=np.mean([1,25])
no_employees=df_clean['no_employees'].tolist()
for i in range(len(no_employees)):
    if no_employees[i] == 'WTD-25' :
        no_employees[i] = f'{int(wtd_n)}-25'
    elif no_employees[i] == '01-WTN':
        no_employees[i] = f'01-{int(wtd_n)}'
    elif no_employees[i] == 'More than 1000':
        no_employees[i] = '>1000'
#checking the unique values in the column after cleaning
print(no_employees)
#replacing the values in the column with the cleaned values
df_clean['no_employees'] = no_employees


This code is used to standardize the values in the no_employees column of the dataset for consistency and easier analysis. Here's what it does:

What It Does:
Inspects all the unique values in the no_employees column to understand the formatting.

Replaces special cases:

'WTD-25': This represents a working time directive range. It is replaced with '13-25', using the mean of 1 and 25 (since '01-WTN' is assumed to represent 1 and 'WTD-25' ends at 25).

'01-WTN': Replaced with '01-13' to reflect the lower half of the same range.

'More than 1000': Simplified to '>1000' for easier comparison and filtering.

Updates the column with these cleaned values.

How to Use It:
Run this block after loading your dataset and before any numeric processing on no_employees.

It prepares the data so you can later:

Convert employee ranges into midpoints or numerical bins.

Visualize company sizes.

Use it in models that need consistent formats.

This step is crucial for turning messy categorical data into clean, interpretable values.

In [ ]:
duplicates = df_clean[df_clean.duplicated()]
#checking for duplicates in the cleaned dataframe
print(duplicates)
#dropping duplicates from the cleaned dataframe
df_clean = df_clean.drop_duplicates()
#checking the cleaned dataframe
df_clean.info()


This code simply checks for and eliminates duplicate rows.

In [ ]:
#converting yes and no to 1 and 0
othervals=["Don't know", 'Not sure', 'Prefer not to say','Some of them', 'Maybe']
df_clean = df_clean.replace({'Yes': 1, 'No': 0 })
columnto_ignore='leave'
for col in df_clean.columns:
    if col != columnto_ignore:
        df_clean[col] = df_clean[col].replace(othervals, -1)
df_clean.head()
#this makes it easier to work with the data in the future


What This Code Does:
Replaces binary responses:

"Yes" <- 1.

"No" <- 0.


Handles ambiguous or unsure responses:

Replaces values like "Don't know", "Not sure", "Prefer not to say", "Some of them", and "Maybe" with -1, in all columns except one (leave). We leave this column out because it contains other values including Dont know and not just yes or no so this will cause inconsistencies in analysis if it was changed

How It Works:
df_clean.replace({'Yes': 1, 'No': 0}): Applies a dictionary mapping to convert yes/no answers globally.

A loop iterates over each column:

for col in df_clean.columns:
    if col != 'leave':
        df_clean[col] = df_clean[col].replace(othervals, -1)
This skips the 'leave' column.

In all other columns, if a value is in the othervals list, it's replaced with -1.

Why This Is Important:

Converting text responses to numeric values makes the data easier to analyze—especially for:

Statistical summaries and visualizations.

Assigning -1 to uncertain answers preserves their presence while clearly distinguishing them from confident responses (0 or 1).

This transformation standardizes your dataset and prepares it for effective, consistent analysis.

In [ ]:
df_clean = df_clean[df_clean['self_employed'].notna()]
df_clean.info()

Dropping Rows with Missing self_employed Values
What This Code Does:
df_clean = df_clean[df_clean['self_employed'].notna()]
This line removes all rows in the dataset where the self_employed column has missing values (NaN).

df_clean.info()
Displays a summary of the DataFrame after cleaning — including row count and column data types — to verify the effect of the change.

How It Works:
df_clean['self_employed'].notna() returns a boolean Series: True for rows with valid values, False for NaN.

Using it inside df_clean[...] filters out the rows with missing values.

The result is reassigned back to df_clean.

Why This Is Done:
Out of 1258 total rows, only 18 rows had missing values in self_employed.

Since that’s a small fraction (~1.4%), dropping them avoids unnecessary complexity and potential bias from imputing or guessing values.

It helps keep the dataset clean and consistent for analysis or modeling, especially if self_employed is important for prediction or grouping.

This step improves data quality without significantly affecting dataset size.

In [ ]:
df_clean.to_csv('cleaned_data.csv', index=False)

saving the cleaned dataset as a csv